## Importing Libraries

In [5]:
import glob
import json
from tqdm import tqdm
import random
import os
from groq import Groq

## Setup Files

In [6]:
GROQ_KEY = os.getenv("GROQ_API_KEY")
PATIENT_PROFILES = glob.glob("./patient_profiles/patient_*.json")
GEN_PROMPT = "./prompts/diary_generation_prompt.txt"
SYS_PROMPT = "./prompts/system_prompt.txt"
DIARY_TEMPLATE = "./diaries_template/diary_template.txt"
DIARY_EXAMPLES = "./diaries_template/diaries_ex.txt"

OUTPUT_DIR = "./outputs/"
OUTPUT_EXP_DIR = "./outputs/diary-gen_experiment"
OUTPUT_FILE = "diary-gen_patient"
TRACK_FILE = "parameter_patient"

MODEL = "llama-3.3-70b-versatile" # llama-3.3-70b-versatile, openai/gpt-oss-120b the prompt isn-t optimized for gpt-oss-120b

## Setup Environment

In [7]:
## Setting evironment
os.makedirs(OUTPUT_DIR,exist_ok=True)

count = 0

for path in os.listdir(OUTPUT_DIR):
    if os.path.isdir(os.path.join(OUTPUT_DIR, path)):
        count += 1
        
os.makedirs(f"{OUTPUT_EXP_DIR}_{count}", exist_ok=True)

## Generating Diaries

In [8]:
client = Groq(api_key=GROQ_KEY)

style_modes = [
    "narrative-dominant",
    "telegraphic-hospital-style",
    "exam-and-imaging-focused",
    "toxicity-focused",
    "psychosocial-emphasis"
]

length_modes = [
    "short",
    "medium",
    "long"
]

temp = 0.7

pbar = tqdm(total=len(PATIENT_PROFILES), desc="Generating sythentic clinical diaries")

for patient in PATIENT_PROFILES:
    print("Processing patient:", patient)
    with open(patient, "r", encoding="utf-8") as f, \
         open(GEN_PROMPT, "r", encoding="utf-8") as gen_prompt_file, \
         open(SYS_PROMPT, "r", encoding="utf-8") as sys_prompt_file, \
         open(DIARY_TEMPLATE, "r", encoding="utf-8") as diary_template_file, \
         open(DIARY_EXAMPLES, "r", encoding="utf-8") as diary_ex_file:
             
        patient_id = patient.split('_')[2].split('.')[0]
        
        selected_style = random.choice(style_modes)
        selected_length = random.choice(length_modes)
        
        print(f"Selected stylistic mode for patient {patient_id}: {selected_style}")
        print(f"Selected length mode for patient {patient_id}: {selected_length}")
        
        patient_data = json.load(f)
        base_gen_prompt = gen_prompt_file.read()
        sys_prompt = sys_prompt_file.read()
        diary_template = diary_template_file.read()
        diary_examples = diary_ex_file.read()
        
        prompt_w_template = base_gen_prompt.replace("{{TEMPLATE_TEXT}}", diary_template)
        prompt_w_patient = prompt_w_template.replace("{{PATIENT_PROF}}", json.dumps(patient_data))
        prompt_w_style = prompt_w_patient.replace("{{STYLISTIC_MODE}}", selected_style)
        prompt_w_length = prompt_w_style.replace("{{LENGTH_MODE}}", selected_length)
        prompt_final = prompt_w_length.replace("{{DIARIES_TEXT}}", diary_examples)
        
        completion = client.chat.completions.create(
            model=MODEL, # llama-3.3-70b-versatile, openai/gpt-oss-120b the prompt isn-t optimized for gpt-oss-120b
            messages=[
                {
                    "role": "system",
                    "content": sys_prompt
                },
                {
                    "role": "user",
                    "content": prompt_final
                }
            ],
            temperature=temp
        )
        result = completion.choices[0].message.content
        
        with open(f"{OUTPUT_EXP_DIR}_{count}/{OUTPUT_FILE}_{patient_id}.txt","w",encoding="utf-8") as o:
            o.write(f"{result}\n\n")
            print(f"Saved LLM output on {OUTPUT_EXP_DIR}_{count}")
        
        with open(f"{OUTPUT_EXP_DIR}_{count}/{TRACK_FILE}_{patient_id}.txt","w",encoding="utf-8") as o:
            o.write(f"Model: {MODEL}\n\n\
                    Stylistic Mode: {selected_style}\n\n\
                    Length Mode: {selected_length}\n\n\
                    Temperature:{temp}\n\n\
                    system prompt: \n{sys_prompt}\n\n\
                    Generation prompt: \n{prompt_final}")
            print(f"Saved parameters to {OUTPUT_EXP_DIR}_{count}")
            
        
        print("\n")
        
        pbar.update(1)
        
pbar.close()

Generating sythentic clinical diaries:  30%|███       | 3/10 [06:07<14:16, 122.41s/it]


Processing patient: ./patient_profiles\patient_1.json
Selected stylistic mode for patient 1: telegraphic-hospital-style
Selected length mode for patient 1: short


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kbn0n3ddf6c8a9npft736hda` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98122, Requested 5642. Please try again in 54m12.096s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}